In [6]:
import numpy as np 
import pandas as pd

In [7]:
hub = pd.read_csv('data/hubs.csv')


In [8]:
raw = pd.read_csv('data/raw_shipments.csv')

In [9]:
new = pd.read_csv('data/new_shipments.csv')

In [10]:
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no"]


In [11]:
len(FEATURES)

17

In [12]:
print("Hello")

Hello


In [14]:
list(hub.columns)

['hub_id', 'hub_city', 'region', 'hub_tier', 'hub_daily_capacity']

In [15]:
list(raw.columns)

['shipment_id',
 'hub_id',
 'service_type',
 'payment_mode',
 'parcel_weight_kg',
 'distance_km',
 'declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'dispatch_time',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no',
 'delivery_hours']

In [18]:
missing_values = raw.isnull().sum()
print("Missing values in raw shipments data:\n", missing_values[missing_values > 0])

Missing values in raw shipments data:
 payment_mode        18
parcel_weight_kg    25
distance_km         36
declared_value      12
dtype: int64


In [19]:
raw.describe()

,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
count,1199.000000,1188.000000,1224.000000,1224.000000,1224.000000,1224.000000,1224.000000,1224.000000,1224.000000,1224.000000,1224.000000
mean,6.477490,337.254461,71.235294,0.177288,0.145425,5526.603758,20.044935,253.741830,0.497979,146.612745,56.953301
std,64.280116,162.977184,15.406942,0.382068,0.352673,2649.101287,11.398330,144.904678,0.289456,87.408686,19.146707
min,0.120000,19.200000,20.000000,0.000000,0.000000,1009.000000,1.000000,1.000000,0.001000,1.000000,16.360000
25%,1.105000,205.400000,60.000000,0.000000,0.000000,3167.000000,10.000000,127.000000,0.250750,70.000000,43.267500
50%,1.830000,314.950000,72.000000,0.000000,0.000000,5568.500000,20.000000,249.000000,0.487500,146.500000,54.435000
75%,3.060000,440.675000,83.000000,0.000000,0.000000,7878.750000,30.000000,380.000000,0.747250,217.250000,72.465000
max,999.000000,990.900000,111.000000,1.000000,1.000000,9983.000000,39.000000,499.000000,1.000000,299.000000,117.180000


In [20]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1224 entries, 0 to 1223
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1224 non-null   object 
 1   hub_id              1224 non-null   object 
 2   service_type        1224 non-null   object 
 3   payment_mode        1206 non-null   object 
 4   parcel_weight_kg    1199 non-null   float64
 5   distance_km         1188 non-null   float64
 6   declared_value      1212 non-null   object 
 7   n_stops             1224 non-null   int64  
 8   rain_flag           1224 non-null   int64  
 9   sale_event_flag     1224 non-null   int64  
 10  dispatch_time       1224 non-null   object 
 11  label_print_batch   1224 non-null   int64  
 12  scanner_build_no    1224 non-null   int64  
 13  tape_roll_no        1224 non-null   int64  
 14  sticker_serial_mod  1224 non-null   float64
 15  shelf_slot_no       1224 non-null   int64  
 16  delive

In [ ]:
len(raw.drop_duplicates())

1200

In [22]:
shipments = raw.drop_duplicates()

In [23]:
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,"₹1,036",90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,"₹1,122",51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,₹288,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10
1198,SHP11199,HUB11,Express,COD,1.71,284.6,"₹4,636",73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55


In [25]:
shipments.to_csv('data/shipments_final.csv', index=False)

In [26]:
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,"₹1,036",90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,"₹1,122",51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,₹288,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10
1198,SHP11199,HUB11,Express,COD,1.71,284.6,"₹4,636",73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55


In [29]:
shipments["service_type"].unique()

array(['express', 'Standard', 'Economy', 'Express', 'STANDARD',
       '  Express ', '  Standard ', '  Economy ', 'ECONOMY', 'EXPRESS',
       'standard', 'economy'], dtype=object)

In [32]:
shipments["service_type"] = (shipments["service_type"].str.strip().str.title())
shipments["service_type"]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_31104\1653591398.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["service_type"] = (shipments["service_type"].str.strip().str.title())


0        Express
1       Standard
2        Economy
3       Standard
4       Standard
          ...   
1195    Standard
1196     Economy
1197     Economy
1198     Express
1199    Standard
Name: service_type, Length: 1200, dtype: object

In [33]:
shipments["service_type"].unique()

array(['Express', 'Standard', 'Economy'], dtype=object)

In [34]:
shipments["service_type"].value_counts()

service_type
Standard    642
Economy     315
Express     243
Name: count, dtype: int64

In [35]:
shipments_df = shipments.copy();

In [36]:
shipments_df

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,Express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,"₹1,036",90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,"₹1,122",51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,₹288,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10
1198,SHP11199,HUB11,Express,COD,1.71,284.6,"₹4,636",73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55


In [37]:
shipments_df.to_csv('data/shipments.csv', index=False)

In [38]:
shipments_df["service_type"].value_counts()

service_type
Standard    642
Economy     315
Express     243
Name: count, dtype: int64

In [40]:
shipments_df["service_type"].value_counts(normalize=True)*100

service_type
Standard    53.50
Economy     26.25
Express     20.25
Name: proportion, dtype: float64

In [43]:
shipments_df["payment_mode"].unique()

array(['COD', 'Prepaid', nan], dtype=object)

In [49]:
shipments_df["payment_mode"]  = (
    shipments_df["payment_mode"].fillna("Unknown")
)

shipments_df["payment_mode"].unique()

array(['COD', 'Prepaid', 'Unknown'], dtype=object)

In [51]:
shipments_df["payment_mode"].value_counts(normalize=True)*100

payment_mode
Prepaid    60.75
COD        37.75
Unknown     1.50
Name: proportion, dtype: float64

In [ ]:
shipments